## Installations

In [ ]:
!pip install -q -U weaviate-client langchain-community langchain langchain-core langchain-text-splitters langchain-google-genai wikipedia langchain_weaviate langchain-huggingface

In [ ]:
from google.colab import userdata
import os
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## Imports

In [ ]:
import weaviate
import requests
from langchain_community.document_loaders import WikipediaLoader



print("Weaviate version:", weaviate.__version__)
print("Imports successful")

/tmp/ipykernel_10180/1583782627.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


Weaviate version: 4.23.1
Imports successful


## Data Loading

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(
    query="Artificial intelligence",
    lang="en",
    load_max_docs=1,
    doc_content_chars_max=100000
)

documents = loader.load()

print("Number of documents:", len(documents))

Number of documents: 1


In [ ]:
print(documents[0])

In [ ]:
print(documents[0].metadata)

{'title': 'Artificial intelligence', 'summary': "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI research

## Text Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 113


In [ ]:
print(chunks[0].page_content)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).


In [ ]:
print(chunks[0].metadata)

{'title': 'Artificial intelligence', 'summary': "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI research

In [ ]:
for i, chunk in enumerate(chunks):
    chunk.metadata = {
        "title": chunk.metadata.get("title", ""),
        "source": chunk.metadata.get("source", ""),
        "chunk_id": i
    }

In [ ]:
chunks[0].metadata

{'title': 'Artificial intelligence',
 'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
 'chunk_id': 0}

## Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # Dim 384
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
test_vector = embedding_model.embed_query(
    "What is artificial intelligence?"
)

print("Vector length:", len(test_vector))
print(test_vector[:10])

Vector length: 384
[-0.015781112015247345, 0.0149086844176054, 0.009095539338886738, 0.03131742402911186, -0.021708223968744278, -0.04852483049035072, 0.04887937754392624, 0.0286731980741024, -0.025597238913178444, 0.0540429912507534]


## Check Client Status

In [ ]:
client = weaviate.connect_to_embedded()

print("Weaviate ready:", client.is_ready())

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 10458


Weaviate ready: True


In [ ]:
from langchain_weaviate import WeaviateVectorStore

In [ ]:
INDEX_NAME = "WikipediaRAG" # Collecion Name

if client.collections.exists(INDEX_NAME):
    print("Clollection is already exists")
    client.collections.delete(INDEX_NAME)

In [ ]:
vector_store = WeaviateVectorStore.from_documents(
documents=chunks,
embedding=embedding_model,
client=client,
index_name=INDEX_NAME,
text_key="text"
)


## Test vector retrieval BEFORE using the LLM

In [ ]:
question = "What are the main applications of artificial intelligence?"

results = vector_store.similarity_search( # vector search
    question,
    k=4
)

print("Retrieved chunks:", len(results))

Retrieved chunks: 4


In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

## Inspection

In [ ]:
question = "What are applications of artificial intelligence?"

retrieved_docs = retriever.invoke(question)

print("Retrieved:", len(retrieved_docs))

Retrieved: 4


## Turn Weaviate into a LangChain Retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
question = "What are applications of artificial intelligence?"

retrieved_docs = retriever.invoke(question)

print("Retrieved:", len(retrieved_docs))

Retrieved: 4


## Connect Gemini LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

## Build a RAG prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are a question-answering assistant.

Answer the question using ONLY the provided context.

If the answer cannot be found in the context,
say:

"I don't know based on the provided Wikipedia article."

Context:
{context}

Question:
{question}

Answer:
"""
)

## Retrieve & Generate

In [ ]:
def ask_rag(question):

    # 1. Retrieve relevant chunks from Weaviate
    retrieved_docs = retriever.invoke(question)

    # 2. Combine chunks into context
    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    # 3. Create prompt
    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    # 4. Ask Gemini
    response = llm.invoke(messages)

    return response, retrieved_docs

In [ ]:
response, sources = ask_rag(
    "What are some applications of artificial intelligence?"
)

print(response.content)

[{'type': 'text', 'text': "Based on the provided context, applications of artificial intelligence include:\n\n* Advanced web search engines (such as Google Search)\n* Chatbots\n* Virtual assistants (such as Siri or Alexa)\n* Autonomous vehicles (including drones, ADAS, and self-driving cars)\n* Play and analysis in strategy games (such as chess and Go) / game-playing systems\n* Content generation (such as text, images, audio, and videos)\n* Targeting online advertisements / targeted advertising (such as AdSense and Facebook)\n* Recommendation systems driving internet traffic (offered by Netflix, YouTube, or Amazon)\n* Automatic language translation (such as Microsoft Translator and Google Translate)\n* Facial recognition (such as Apple's FaceID, Facebook's DeepFace, and Google's FaceNet)\n* Image labelling (used by Facebook, Apple's Photos, and TikTok)\n* Industrial robotics", 'extras': {'signature': 'EqUTCqITAWkUfRNa3qJ28e2Z0Ej47oCnY6XJwev6aKxEZ6Ot9/hhXre1gTrt+zDNVbFp27P2prFF+MLodnS5S

In [ ]:
for i, doc in enumerate(sources, start=1):

    print(f"\nSOURCE {i}")
    print("Title:", doc.metadata.get("title"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print("URL:", doc.metadata.get("source"))
    print()
    print(doc.page_content[:700])


SOURCE 1
Title: Artificial intelligence
Chunk: 0.0
URL: https://en.wikipedia.org/wiki/Artificial_intelligence

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g.

SOURCE 2
Title: Artificial intelligence
Chunk: 28.0
URL: https://en.wikipedia.org/wiki/Artificial_intelligence

== Applications ==

AI and machine learning technology is used in most of 